# 02_silver: Clean Bronze data into Silver Delta tables

**Input:** Bronze files in `bronze/<table>/ingestion_date=<date>/`
**Output:** 6 Delta tables in `silver/<table>` (overwrite → idempotent re-runs)
**Parameter:** `ingestion_date` (widget, passed by ADF; defaults to 2026-09-23)

**Pattern per table:** read → clean → tracking columns → quality asserts → write → read-back count

**Conventions:** special keys in UPPER CASE (`UNKNOWN`, `GUEST`); descriptive placeholders "Unknown"; money as decimal; `is_future_dated` compared against the ingestion date (deterministic)

In [0]:
from pyspark.sql import functions as F
# Storage locations (ADLS Gen2 containers)
BRONZE_PATH = "abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@ecommercedatalakebp01.dfs.core.windows.net"
# Which Bronze folder to read: set by ADF at run time, default for manual runs
dbutils.widgets.text("ingestion_date", "2026-09-23")
INGESTION_DATE = dbutils.widgets.get("ingestion_date")

In [0]:
# Quality gate: fail the run if any listed column contains nulls
def assert_no_nulls(df, cols, table):
    for c in cols:
        n = df.filter(F.col(c).isNull()).count()
        assert n == 0, f"{table}: {n} nulls in {c}"

## Customers
- Null `city` / `email` → "Unknown" (descriptive placeholder; customers are kept, not dropped)
- `signup_date` string → date (strict `to_date`: unexpected formats fail the run)
- Asserts: `customer_id` unique; no nulls left in cleaned columns

In [0]:
path_customers=f"{BRONZE_PATH}/customers/ingestion_date={INGESTION_DATE}/customers.parquet"
df_customers=spark.read.parquet(path_customers)

In [0]:
df_customers_clean = (
    df_customers
    .fillna({"email": "Unknown", "city": "Unknown"})
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))
)

In [0]:
df_customers_silver = (
    df_customers_clean
    .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
assert df_customers_silver.select("customer_id").distinct().count() == df_customers_silver.count(), "customers: duplicate customer_id"
assert_no_nulls(df_customers_silver, ["city", "email", "signup_date"], "customers")

In [0]:
customers_silver_path = f"{SILVER_PATH}/customers"
(df_customers_silver.write.mode("overwrite").format("delta")
    .save(customers_silver_path))

In [0]:
check = spark.read.format("delta").load(customers_silver_path)
print("rows in silver:", check.count())


## Products
- `price` string → decimal(10,2) (money is never stored as double)
- One product had a negative price (the root cause of negatives in orders/payments/returns) → `price_sign_corrected` flag set **before** `abs()`
- 2 null prices kept null on purpose (unknown, not invented)

In [0]:
products_path = f"{BRONZE_PATH}/products/ingestion_date={INGESTION_DATE}/products.csv"
df_products = spark.read.format("csv").option("header", "True").load(products_path)

In [0]:
df_products_fixed = (
    df_products
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    # flag BEFORE abs(), otherwise every row would look "not corrected"
    .withColumn("price_sign_corrected", F.when(F.col("price") < 0, True).otherwise(False))
    .withColumn("price", F.abs(F.col("price")))
)

In [0]:
df_products_silver=(df_products_fixed
                    .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
                    .withColumn("_processed_at",F.current_timestamp()))

In [0]:
assert df_products_silver.select("product_id").distinct().count() == df_products_silver.count(), "products: duplicate product_id"
assert df_products_silver.filter(F.col("price") < 0).count() == 0, "products: negative price left after fix"

In [0]:
products_silver_path=f"{SILVER_PATH}/products"
(df_products_silver.write.mode("overwrite").format("delta").save(products_silver_path))


In [0]:
print("rows in silver:", spark.read.format("delta").load(products_silver_path).count())

## Orders
- 3 exact duplicate rows (proven in profiling: distinct whole rows = distinct order_id) → whole-row `dropDuplicates()`; the uniqueness assert catches any non-identical duplicates
- 3 null `customer_id` → "UNKNOWN" (matches the unknown-member row in Gold `dim_customers`)
- Types: `order_date` → date, `price` → decimal(10,2), `revenue` → decimal(12,2)
- 167 negative amounts (165 from the bad product price + 2 isolated sign errors) → `amount_sign_corrected` flag before `abs()`

In [0]:
orders_path=f"{BRONZE_PATH}/orders/ingestion_date={INGESTION_DATE}/"
df_orders=spark.read.format("parquet").load(orders_path)

In [0]:
print("total rows:", df_orders.count())
print("distinct whole rows:", df_orders.distinct().count())

In [0]:
df_orders_fixed = (
    df_orders
    .dropDuplicates()
    .fillna({"customer_id": "UNKNOWN"})
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("revenue", F.col("revenue").cast("decimal(12,2)"))
    # flag BEFORE abs()
    .withColumn("amount_sign_corrected", F.when((F.col("price") < 0) | (F.col("revenue") < 0), True).otherwise(False))
    .withColumn("price", F.abs(F.col("price")))
    .withColumn("revenue", F.abs(F.col("revenue")))
)

In [0]:
df_orders_silver=(df_orders_fixed
                  .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
                  .withColumn("_processed_at", F.current_timestamp()))


In [0]:
orders_silver_path = f"{SILVER_PATH}/orders"
df_orders_silver.write.mode("overwrite").format("delta").save(orders_silver_path)

In [0]:
assert df_orders_silver.select("order_id").distinct().count() == df_orders_silver.count(), "orders: duplicate order_id"
assert_no_nulls(df_orders_silver, ["customer_id", "order_date"], "orders")
assert df_orders_silver.filter((F.col("price") < 0) | (F.col("revenue") < 0)).count() == 0, "orders: negative amount left after fix"

In [0]:
print("rows in silver:", spark.read.format("delta").load(orders_silver_path).count())

## Payments
- `payment_date` string → date; `payment_amount` double → decimal(12,2)
- 165 negative amounts (bad product price) → `amount_sign_corrected` flag before `abs()`
- 2 null amounts kept null (unknown, not invented)
- `is_future_dated` = payment_date > ingestion date (deterministic, not `current_date()`)

In [0]:
payments_path=f"{BRONZE_PATH}/payments/ingestion_date={INGESTION_DATE}/"
df_payments= spark.read.format("json").load(payments_path)


In [0]:
df_payments_fixed = (
    df_payments
    .withColumn("payment_date", F.to_date(F.col("payment_date"), "yyyy-MM-dd"))
    .withColumn("payment_amount", F.col("payment_amount").cast("decimal(12,2)"))
    # flag BEFORE abs()
    .withColumn("amount_sign_corrected", F.when(F.col("payment_amount") < 0, True).otherwise(False))
    .withColumn("payment_amount", F.abs(F.col("payment_amount")))
    .withColumn("is_future_dated", F.col("payment_date") > F.to_date(F.lit(INGESTION_DATE)))
)

In [0]:
df_payments_silver=(df_payments_fixed
                 .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
                 .withColumn("_processed_at", F.current_timestamp()))

In [0]:
assert df_payments_silver.select("payment_id").distinct().count() == df_payments_silver.count(), "payments: duplicate payment_id"
assert_no_nulls(df_payments_silver, ["payment_date"], "payments")
assert df_payments_silver.filter(F.col("payment_amount") < 0).count() == 0, "payments: negative amount left after fix"

In [0]:
df_payments_silver.write.mode("overwrite").format("delta").save(f"{SILVER_PATH}/payments")

In [0]:
print("rows in silver:", spark.read.format("delta").load(f"{SILVER_PATH}/payments").count())

## Returns
- 2 null `return_reason` → "Unknown" (descriptive placeholder)
- `return_date` string → date; `refund_amount` → decimal(12,2) (a refund can equal a whole order's value)
- 22 negative refunds (bad product price) → `amount_sign_corrected` flag before `abs()`
- `is_future_dated` = return_date > ingestion date (deterministic)

In [0]:
returns_path = f"{BRONZE_PATH}/returns/ingestion_date={INGESTION_DATE}/"
df_returns = spark.read.option("header", True).csv(returns_path)


In [0]:
df_returns_fixed = (
    df_returns
    .fillna({"return_reason": "Unknown"})
    .withColumn("return_date", F.to_date(F.col("return_date"), "yyyy-MM-dd"))
    .withColumn("refund_amount", F.col("refund_amount").cast("decimal(12,2)"))
    # flag BEFORE abs()
    .withColumn("amount_sign_corrected", F.when(F.col("refund_amount") < 0, True).otherwise(False))
    .withColumn("refund_amount", F.abs(F.col("refund_amount")))
    .withColumn("is_future_dated", F.col("return_date") > F.to_date(F.lit(INGESTION_DATE)))
)

In [0]:
df_returns_silver=(df_returns_fixed
                 .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
                 .withColumn("_processed_at", F.current_timestamp()))


In [0]:
assert df_returns_silver.select("return_id").distinct().count() == df_returns_silver.count(), "returns: duplicate return_id"
assert_no_nulls(df_returns_silver, ["return_reason", "return_date"], "returns")
assert df_returns_silver.filter(F.col("refund_amount") < 0).count() == 0, "returns: negative refund left after fix"

In [0]:
df_returns_silver.write.mode("overwrite").format("delta").save(f"{SILVER_PATH}/returns")
print("rows in silver:", spark.read.format("delta").load(f"{SILVER_PATH}/returns").count())

## Website events
- Grain: one row = one customer action (clickstream)
- 2 null `customer_id` → "GUEST" (key value; matches the GUEST row in Gold `dim_customers`)
- `event_timestamp` string → timestamp (`yyyy-MM-dd HH:mm:ss`)
- `is_future_dated` compares the **date part** of the timestamp with the ingestion date (same granularity)
- 2 `unknown_event` rows kept here (faithful record); filtered out in Gold

In [0]:
events_path = f"{BRONZE_PATH}/website_events/ingestion_date={INGESTION_DATE}/"
df_events = spark.read.option("header", True).csv(events_path)

In [0]:
df_events_fixed = (
    df_events
    .fillna({"customer_id": "GUEST"})
    .withColumn("event_timestamp", F.to_timestamp(F.col("event_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    # compare dates with dates: an event at 22:00 on the ingestion day is NOT future
    .withColumn("is_future_dated", F.to_date(F.col("event_timestamp")) > F.to_date(F.lit(INGESTION_DATE)))
)

In [0]:
df_events_silver=(df_events_fixed
                 .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
                 .withColumn("_processed_at", F.current_timestamp()))

In [0]:
assert df_events_silver.select("event_id").distinct().count() == df_events_silver.count(), "website_events: duplicate event_id"
assert_no_nulls(df_events_silver, ["customer_id", "event_timestamp"], "website_events")

In [0]:
df_events_silver.write.mode("overwrite").format("delta").save(f"{SILVER_PATH}/website_events")
print("rows in silver:", spark.read.format("delta").load(f"{SILVER_PATH}/website_events").count())